# Train of the classifier
In this file we make deep analysis of the impact of the subspace alignment method to the task of classification.

We perform the following scenarios:
1) we train an MLP using the full dimensions of the original space
2) we comput the alignment matrix using training set and we use the subspace alignment method to align the subspaces. We set the subspace dim to 256 as it is a good result in terms of compression/performance.
Then we train the MLP and we make it accept just the fused vector (See fusion strategy) composed by all the dimensions in the aligned space.
3) we select, using the alignment matrix, rhe most important dimension and then we fit the mlp on those dimension taken in the alignment space
4) we select, using the alignment matrix, rhe most important dimension and then we fit the mlp on those dimension taken in the original space
5) we run with the same number of dimensions from before but taken both randomly and first-k


For each experiment we record the accuracy, we trai using cross entropy loss and we track the 3 types of gaps, namely RMG, L2M and L2I.

In [1]:
import sys
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader    
import os
import sys

# os.chdir(os.path.dirname(os.path.abspath(__file__)))
sys.path.append(os.path.abspath(".."))

from analysis.viz import visualize_3d, tsne_3d
from metrics.retrieval import retrieval
from dataset.dataloader_embeddings_with_labels import EmbeddingsDatasetWithLabels
from analysis.modality_gap import compute_gap
import procrustes
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from dataset.flickr_imagenet_utils import original_idx2class
from models.fusion_mlp_classifier import LinearProbing
# from models.fusion_mlp_classifier import FusionMLPClassifier
import torch.nn.functional as F
import torch 
import torch.nn as nn
import wandb
import copy
from torch.nn.utils import clip_grad_norm_
# Add parent directory to Python path so we can import `models.*`
seed = 123
g = torch.Generator().manual_seed(seed)

In [2]:
# Let's get the data
precomputed_dir = "/mnt/media/emanuele/few_dimensions/dataset/precomputed_embeddings_with_labels/clip_vit_b_32___laion2b_s34b_b79k"
dataset = EmbeddingsDatasetWithLabels(precomputed_dir=precomputed_dir, split_name="flickr30k")

[Loaded] 31783 samples from /mnt/media/emanuele/few_dimensions/dataset/precomputed_embeddings_with_labels/clip_vit_b_32___laion2b_s34b_b79k


## Filter out data construct dataloaders
We need to filter out the classes with very few samples, this may make the classifier struggle.

In [3]:
# We should make sure to filter out classes that have too few samples, otherwise the classifier will struggle to learn. 
# Let's set a threshold of 1% of dataset total length as samples per class constraint.
# threshold = int(0.01 * len(dataset))
threshold = 10
print(f"Filtering out classes with less than {threshold} samples...")
class_counts = {}
for _, _, label in dataset:
    if label.item() not in class_counts:
        class_counts[label.item()] = 0
    class_counts[label.item()] += 1
    
filtered_classes = {cls for cls, count in class_counts.items() if count >= threshold}
print(f"Number of classes after filtering: {len(filtered_classes)}")
filtered_indices = [i for i, (_,_, label) in enumerate(dataset) if label.item() in filtered_classes]
filtered_dataset = torch.utils.data.Subset(dataset, filtered_indices)
print(f"Number of samples after filtering: {len(filtered_dataset)}")

Filtering out classes with less than 10 samples...
Number of classes after filtering: 465
Number of samples after filtering: 30471


In [4]:
# sort the class count by count value
class_counts = {cls: count for cls, count in class_counts.items() if cls in filtered_classes}
class_counts = dict(sorted(class_counts.items(), key=lambda item: item[1], reverse=True))
# print("Class counts after filtering:")
# for cls, count in class_counts.items():
#     print(f"Class {cls}: {count} samples")  

In [5]:
# Now split the filtered dataset into train and test only (stratified).
# We'll use a 80-20 split for train and test.

from sklearn.model_selection import train_test_split

# Extract labels for the filtered dataset
filtered_labels = []
for i in filtered_indices:
    _, _, label = dataset[i]
    filtered_labels.append(label.item())

filtered_labels = np.array(filtered_labels)

# Indices relative to filtered_dataset
all_indices = np.arange(len(filtered_dataset))

# Stratified split: Train (80%) and Test (20%)
train_idx, test_idx, _, _ = train_test_split(
    all_indices,
    filtered_labels,
    test_size=0.20,
    stratify=filtered_labels,
    random_state=seed
)

# Create final subsets
train_dataset = torch.utils.data.Subset(filtered_dataset, train_idx)
test_dataset = torch.utils.data.Subset(filtered_dataset, test_idx)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")


Train size: 24376
Test size: 6095


In [6]:
# check the class distribution in each split
def get_class_distribution(subset):
    class_counts = {}
    for _, _, label in subset:
        if label.item() not in class_counts:
            class_counts[label.item()] = 0
        class_counts[label.item()] += 1
    return class_counts

train_class_dist = get_class_distribution(train_dataset)
test_class_dist = get_class_distribution(test_dataset)

print("Number of classes in train set:", len(train_class_dist))
print("Number of classes in test set:", len(test_class_dist))
    

Number of classes in train set: 465
Number of classes in test set: 465


In [7]:
len_classes = len(train_class_dist)

# build a mapping from the original class indices to the new class indices after filtering
filtered_idx2class = {idx: val for idx, val in enumerate(sorted(train_class_dist.keys()))}
filtered_class2idx = {v: k for k, v in filtered_idx2class.items()}
new_idx2class = {idx: original_idx2class[val] for idx, val in filtered_idx2class.items()}
new_class2idx = {v: k for k, v in new_idx2class.items()}


In [8]:
# Now we can build the dataloaders for each split.
bs = 2048
train_dataloader = DataLoader(train_dataset, batch_size=bs, shuffle=True, generator=g)
test_dataloader = DataLoader(test_dataset, batch_size=bs, shuffle=False, generator=g)

## Scenario 1
Train the MLP on the full dimension of the origianal space.
Track accuracy, for train and validation, loss and gap.

In [10]:
d = train_dataset[0][0].shape[0]
# Set the config for this scenario
config = {
    'classifier': {
    'd' : d,
    'num_classes': len_classes,
    },
    'epochs' : 1000,
    'batch_size' : bs,
    'wandb': {
        'enabled': True,
        'project': 'flickr30k_clip_classifier',
        'name': 'scenario_1',
    }
}

In [11]:
# ---- optimizer + scheduler ----
def make_optimizer_and_scheduler(model, train_loader, epochs=20, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler

def gap_to_float(x):
    if isinstance(x, dict):
        if "text_vision" in x:
            return float(x["text_vision"])
        return float(next(iter(x.values())))
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)


In [ ]:
# =========================
# TRAIN ONLY on cuda:3
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Save paths
# -------------------------
save_dir = "scenario_1"
os.makedirs(save_dir, exist_ok=True)

best_acc_path  = os.path.join(save_dir, "best_model_val_acc.pt")
best_loss_path = os.path.join(save_dir, "best_model_val_loss.pt")

# -------------------------
# Model
# -------------------------
model_wrapper = LinearProbing(
    d=config["classifier"]["d"],
    num_classes=config["classifier"]["num_classes"],
).to(device)

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = int(config["epochs"])
gaps = ["RMG", "L2M", "L2I", "cosineTP"]  # add cosineTP to the list of gaps to compute

# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0         # improvement if val_loss < best - min_delta_loss
min_delta_acc  = 0.0         # improvement if val_acc  > best + min_delta_acc
epochs_no_improve = 0

# -------------------------
# W&B (optional)
# -------------------------
use_wandb = bool(config.get("wandb", {}).get("enabled", False))
if use_wandb:
    import wandb
    if wandb.run is None:
        wandb.init(
            project=config["wandb"].get("project", "classifier"),
            name=config["wandb"].get("run_name", None),
            config=config,
        )

# -------------------------
# Optimizer + Scheduler
# -------------------------
optimizer, scheduler = make_optimizer_and_scheduler(
    model_wrapper,
    train_dataloader,
    epochs=epochs,
    max_lr=3e-4,
    weight_decay=1e-4
)

# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped

# -------------------------
# Helpers
# -------------------------
def gap_to_float(x):
    # supports dict + scalar tensors
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)
    
def run_epoch(dataloader, phase: str):
    """
    phase: "train" or "val"
    Returns: mean_loss, mean_acc, mean_lr (train only else None), mean_gaps dict
    """
    is_train = (phase == "train")
    if is_train:
        model_wrapper.train()
    else:
        model_wrapper.eval()

    losses, accs, lrs = [], [], []
    gaps_accum = {g: [] for g in gaps}

    for batch in tqdm(dataloader, desc=f"{phase.upper()}"):
        text_emb, vision_emb, labels = batch

        text_emb = text_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        vision_emb = vision_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        labels = labels.to(device=device, dtype=torch.long, non_blocking=True)
        labels = remap_labels(labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model_wrapper(text_emb, vision_emb)

        # Safety: label range
        num_classes = logits.size(1)
        y_min = int(labels.min().item())
        y_max = int(labels.max().item())
        if y_min < 0 or y_max >= num_classes:
            raise ValueError(
                f"[{phase.upper()}] Labels out of range after remap: "
                f"min={y_min}, max={y_max}, num_classes={num_classes}"
            )

        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            clip_grad_norm_(model_wrapper.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        acc = (logits.argmax(dim=1) == labels).float().mean()

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        if is_train:
            lrs.append(float(optimizer.param_groups[0]["lr"]))

        # ---- GAPs on GPU ----
        text_det = text_emb.detach()
        vis_det  = vision_emb.detach()
        for g in gaps:
            gaps_accum[g].append(gap_to_float(compute_gap(g, text_det, vis_det, iterations=None)))

    mean_loss = float(np.mean(losses)) if losses else float("nan")
    mean_acc  = float(np.mean(accs)) if accs else float("nan")
    mean_lr   = float(np.mean(lrs)) if (is_train and lrs) else None
    mean_gaps = {g: float(np.mean(gaps_accum[g])) for g in gaps}

    return mean_loss, mean_acc, mean_lr, mean_gaps

# -------------------------
# Tracking
# -------------------------
epochs_results = {
    "train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
    "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []},
}

best_val_loss = float("inf")
best_loss_model_state_dict = None

best_val_acc = 0.0
best_acc_model_state_dict = None

# -------------------------
# Training loop
# -------------------------
for epoch in range(epochs):
    # ---- TRAIN ----
    train_loss, train_acc, train_lr, train_gaps = run_epoch(train_dataloader, phase="train")

    epochs_results["train"]["loss"].append(train_loss)
    epochs_results["train"]["accuracy"].append(train_acc)
    epochs_results["train"]["lr"].append(train_lr if train_lr is not None else float("nan"))
    for g in gaps:
        epochs_results["train"][g].append(train_gaps[g])

    # ---- VAL ----
    with torch.no_grad():   
        val_loss, val_acc, _, val_gaps = run_epoch(test_dataloader, phase="val")

    epochs_results["val"]["loss"].append(val_loss)
    epochs_results["val"]["accuracy"].append(val_acc)
    for g in gaps:
        epochs_results["val"][g].append(val_gaps[g])

    # ---- Logging ----
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} lr={train_lr:.6f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )
        print(
            f"   train gaps: RMG={train_gaps['RMG']:.4f}, L2M={train_gaps['L2M']:.4f}, L2I={train_gaps['L2I']:.4f}, cosineTP={train_gaps['cosineTP']:.4f}"
        )
        print(
            f"   val   gaps: RMG={val_gaps['RMG']:.4f}, L2M={val_gaps['L2M']:.4f}, L2I={val_gaps['L2I']:.4f}, cosineTP={val_gaps['cosineTP']:.4f}"
        )

    # -------------------------
    # Early stopping based on VALIDATION improvements
    # "Improve" if EITHER val_loss improves OR val_acc improves
    # -------------------------
    improved = False

    # Best-by-VAL LOSS
    if val_loss < (best_val_loss - min_delta_loss):
        best_val_loss = val_loss
        best_loss_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
        improved = True

        torch.save(
            {
                "epoch": epoch + 1,
                "metric": "val_loss",
                "metric_value": best_val_loss,
                "model_state_dict": best_loss_model_state_dict,
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config,
            },
            best_loss_path,
        )
        print(f"[BEST VAL LOSS] epoch={epoch+1} val_loss={best_val_loss:.4f} -> saved: {best_loss_path}")

    # Best-by-VAL ACC
    if val_acc > (best_val_acc + min_delta_acc):
        best_val_acc = val_acc
        best_acc_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
        improved = True

        torch.save(
            {
                "epoch": epoch + 1,
                "metric": "val_accuracy",
                "metric_value": best_val_acc,
                "model_state_dict": best_acc_model_state_dict,
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config,
            },
            best_acc_path,
        )
        print(f"[BEST VAL ACC]  epoch={epoch+1} val_acc={best_val_acc:.4f} -> saved: {best_acc_path}")

    if improved:
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"[EARLY STOP CHECK] no val improvement (loss or acc): {epochs_no_improve}/{patience}")

    if use_wandb:
        wandb.log({
            "epoch": epoch + 1,

            "train/loss": train_loss,
            "train/acc": train_acc,
            "train/lr": train_lr,
            "train/RMG": train_gaps["RMG"],
            "train/L2M": train_gaps["L2M"],
            "train/L2I": train_gaps["L2I"],
            "train/cosineTP": train_gaps["cosineTP"],
            "val/loss": val_loss,
            "val/acc": val_acc,
            "val/RMG": val_gaps["RMG"],
            "val/L2M": val_gaps["L2M"],
            "val/L2I": val_gaps["L2I"],
            "val/cosineTP": val_gaps["cosineTP"],

            "early_stop/epochs_no_improve": epochs_no_improve,
            "best/val_loss": best_val_loss,
            "best/val_acc": best_val_acc,
        })

    if epochs_no_improve >= patience:
        print(f"\n[EARLY STOP] Stopped at epoch {epoch+1} (patience={patience}) based on VALIDATION.")
        break

print("\nTraining finished.")
print("Best val loss:", best_val_loss)
print("Best val acc :", best_val_acc)

# -------------------------
# Load best checkpoint (choose one)
# -------------------------
# Default: load best VAL ACC model (common for classification)
if best_acc_model_state_dict is not None:
    model_wrapper.load_state_dict(best_acc_model_state_dict)
    print("Loaded best model by VAL ACC.")
elif best_loss_model_state_dict is not None:
    model_wrapper.load_state_dict(best_loss_model_state_dict)
    print("Loaded best model by VAL LOSS.")
else:
    print("No best model saved (unexpected).")

final_best_path = os.path.join(save_dir, "best_model_loaded_final.pt")
torch.save(
    {
        "model_state_dict": model_wrapper.state_dict(),
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "config": config,
    },
    final_best_path,
)
print(f"Final loaded best model saved to: {final_best_path}")

## Scenario 2
We fit the linear probing layer on the entire aligned space. (Set 256 as the subspace dimension)

In [9]:
from subspace_alignment.subspace_alignment import apply_subspace_alignment, fit_subspace_alignment, fit_subspace_alignment
with torch.no_grad():
    sub_model = fit_subspace_alignment(train_dataloader, d_sub=256, device="cuda")

Collected 24376 samples of dimension 512


In [10]:
d = train_dataset[0][0].shape[0]
# Set the config for this scenario
config = {
    'classifier': {
    'd' : d,
    'num_classes': len_classes,
    },
    'epochs' : 1000,
    'batch_size' : bs,
    'wandb': {
        'enabled': True,
        'project': 'flickr30k_clip_classifier',
        'name': 'scenario_2',
    }
}

In [11]:
# ---- optimizer + scheduler ----
def make_optimizer_and_scheduler(model, train_loader, epochs=20, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler

def gap_to_float(x):
    if isinstance(x, dict):
        if "text_vision" in x:
            return float(x["text_vision"])
        return float(next(iter(x.values())))
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)


In [ ]:
# =========================
# TRAIN ONLY on cuda:3
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Save paths
# -------------------------
save_dir = config['wandb']['name']
os.makedirs(save_dir, exist_ok=True)
best_acc_path  = os.path.join(save_dir, "best_model_val_acc.pt")
best_loss_path = os.path.join(save_dir, "best_model_val_loss.pt")

# -------------------------
# Model
# -------------------------
model_wrapper = LinearProbing(
    d=config["classifier"]["d"],
    num_classes=config["classifier"]["num_classes"],
).to(device)

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = int(config["epochs"])
gaps = ["RMG", "L2M", "L2I", "cosineTP"]  # add cosineTP to the list of gaps to compute


# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0         # improvement if val_loss < best - min_delta_loss
min_delta_acc  = 0.0         # improvement if val_acc  > best + min_delta_acc
epochs_no_improve = 0

# -------------------------
# W&B (optional)
# -------------------------
use_wandb = bool(config.get("wandb", {}).get("enabled", False))
if use_wandb:
    import wandb
    if wandb.run is None:
        wandb.init(
            project=config["wandb"].get("project", "classifier"),
            name='scenario_2',
            config=config,
        )
        
# -------------------------
# Optimizer + Scheduler
# -------------------------
optimizer, scheduler = make_optimizer_and_scheduler(
    model_wrapper,
    train_dataloader,
    epochs=epochs,
    max_lr=3e-4,
    weight_decay=1e-4
)


# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped

# -------------------------
# Helpers
# -------------------------
def gap_to_float(x):
    # supports dict + scalar tensors
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/emanuele/.netrc.


Using device: cuda:3 - NVIDIA RTX A6000


wandb: Currently logged in as: rucci-emanuele (rucci-emanuele-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
def run_epoch(dataloader, phase: str):
    """
    phase: "train" or "val"
    Returns: mean_loss, mean_acc, mean_lr (train only else None), mean_gaps dict
    """
    is_train = (phase == "train")
    if is_train:
        model_wrapper.train()
    else:
        model_wrapper.eval()

    losses, accs, lrs = [], [], []
    gaps_accum = {g: [] for g in gaps}

    for batch in tqdm(dataloader, desc=f"{phase.upper()}"):
        text_emb, vision_emb, labels = batch

        text_emb = text_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        vision_emb = vision_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        
        # Subspace Alignment
        with torch.no_grad():
            X, _, Y_al = apply_subspace_alignment(text_emb, vision_emb, sub_model, device=device, renorm=True)
            X = X.to(device)
            Y_al = Y_al.to(device)
            
        labels = labels.to(device=device, dtype=torch.long, non_blocking=True)
        labels = remap_labels(labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model_wrapper(X, Y_al)

        # Safety: label range
        num_classes = logits.size(1)
        y_min = int(labels.min().item())
        y_max = int(labels.max().item())
        if y_min < 0 or y_max >= num_classes:
            raise ValueError(
                f"[{phase.upper()}] Labels out of range after remap: "
                f"min={y_min}, max={y_max}, num_classes={num_classes}"
            )

        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            clip_grad_norm_(model_wrapper.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        acc = (logits.argmax(dim=1) == labels).float().mean()

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        if is_train:
            lrs.append(float(optimizer.param_groups[0]["lr"]))

        # ---- GAPs on GPU USING ALIGNED REPRESENTATIONS ----
        text_det = X.detach()
        vis_det  = Y_al.detach()
        for g in gaps:
            gaps_accum[g].append(gap_to_float(compute_gap(g, text_det, vis_det, iterations=None)))

    mean_loss = float(np.mean(losses)) if losses else float("nan")
    mean_acc  = float(np.mean(accs)) if accs else float("nan")
    mean_lr   = float(np.mean(lrs)) if (is_train and lrs) else None
    mean_gaps = {g: float(np.mean(gaps_accum[g])) for g in gaps}

    return mean_loss, mean_acc, mean_lr, mean_gaps


In [ ]:
# -------------------------
# Tracking
# -------------------------
epochs_results = {
    "train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
    "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []},
}

best_val_loss = float("inf")
best_loss_model_state_dict = None

best_val_acc = 0.0
best_acc_model_state_dict = None

# -------------------------
# Training loop
# -------------------------
for epoch in range(epochs):
    # ---- TRAIN ----
    train_loss, train_acc, train_lr, train_gaps = run_epoch(train_dataloader, phase="train")

    epochs_results["train"]["loss"].append(train_loss)
    epochs_results["train"]["accuracy"].append(train_acc)
    epochs_results["train"]["lr"].append(train_lr if train_lr is not None else float("nan"))
    for g in gaps:
        epochs_results["train"][g].append(train_gaps[g])

    # ---- VAL ----
    with torch.no_grad():   
        val_loss, val_acc, _, val_gaps = run_epoch(test_dataloader, phase="val")

    epochs_results["val"]["loss"].append(val_loss)
    epochs_results["val"]["accuracy"].append(val_acc)
    for g in gaps:
        epochs_results["val"][g].append(val_gaps[g])

    # ---- Logging ----
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} lr={train_lr:.6f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )
        print(
            f"   train gaps: RMG={train_gaps['RMG']:.4f}, L2M={train_gaps['L2M']:.4f}, L2I={train_gaps['L2I']:.4f}, cosineTP={train_gaps['cosineTP']:.4f}"
        )
        print(
            f"   val   gaps: RMG={val_gaps['RMG']:.4f}, L2M={val_gaps['L2M']:.4f}, L2I={val_gaps['L2I']:.4f}, cosineTP={val_gaps['cosineTP']:.4f}"
        )

    # -------------------------
    # Early stopping based on VALIDATION improvements
    # "Improve" if EITHER val_loss improves OR val_acc improves
    # -------------------------
    improved = False

    # Best-by-VAL LOSS
    if val_loss < (best_val_loss - min_delta_loss):
        best_val_loss = val_loss
        best_loss_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
        improved = True

        torch.save(
            {
                "epoch": epoch + 1,
                "metric": "val_loss",
                "metric_value": best_val_loss,
                "model_state_dict": best_loss_model_state_dict,
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config,
            },
            best_loss_path,
        )
        print(f"[BEST VAL LOSS] epoch={epoch+1} val_loss={best_val_loss:.4f} -> saved: {best_loss_path}")

    # Best-by-VAL ACC
    if val_acc > (best_val_acc + min_delta_acc):
        best_val_acc = val_acc
        best_acc_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
        improved = True

        torch.save(
            {
                "epoch": epoch + 1,
                "metric": "val_accuracy",
                "metric_value": best_val_acc,
                "model_state_dict": best_acc_model_state_dict,
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config,
            },
            best_acc_path,
        )
        print(f"[BEST VAL ACC]  epoch={epoch+1} val_acc={best_val_acc:.4f} -> saved: {best_acc_path}")

    if improved:
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"[EARLY STOP CHECK] no val improvement (loss or acc): {epochs_no_improve}/{patience}")

    if use_wandb:
        wandb.log({
            "epoch": epoch + 1,

            "train/loss": train_loss,
            "train/acc": train_acc,
            "train/lr": train_lr,
            "train/RMG": train_gaps["RMG"],
            "train/L2M": train_gaps["L2M"],
            "train/L2I": train_gaps["L2I"],
            "train/cosineTP": train_gaps["cosineTP"],
            "val/loss": val_loss,
            "val/acc": val_acc,
            "val/RMG": val_gaps["RMG"],
            "val/L2M": val_gaps["L2M"],
            "val/L2I": val_gaps["L2I"],
            "val/cosineTP": val_gaps["cosineTP"],

            "early_stop/epochs_no_improve": epochs_no_improve,
            "best/val_loss": best_val_loss,
            "best/val_acc": best_val_acc,
        })

    if epochs_no_improve >= patience:
        print(f"\n[EARLY STOP] Stopped at epoch {epoch+1} (patience={patience}) based on VALIDATION.")
        break

print(f"\nTraining finished.")
print("Best val loss:", best_val_loss)
print("Best val acc :", best_val_acc)

# -------------------------
# Load best checkpoint (choose one)
# -------------------------
# Default: load best VAL ACC model (common for classification)
if best_acc_model_state_dict is not None:
    model_wrapper.load_state_dict(best_acc_model_state_dict)
    print("Loaded best model by VAL ACC.")
elif best_loss_model_state_dict is not None:
    model_wrapper.load_state_dict(best_loss_model_state_dict)
    print("Loaded best model by VAL LOSS.")
else:
    print("No best model saved (unexpected).")

final_best_path = os.path.join(save_dir, "best_model_loaded_final.pt")
torch.save(
    {
        "model_state_dict": model_wrapper.state_dict(),
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "config": config,
    },
    final_best_path,
)
print(f"Final loaded best model saved to: {final_best_path}")


## Scenario 3.1
We fit the linear probing layer on the first k dims by joint-importance selected by considering the subspace dimension = 256.

joint importance = k3 <br>
linear_probing(cat(text[k3], vision[k3]))



In [15]:
from subspace_alignment.subspace_alignment import fit_subspace_alignment, fit_subspace_alignment, analyze_subspace_dimensions, apply_subspace_alignment
with torch.no_grad():
    sub_model = fit_subspace_alignment(train_dataloader, d_sub=256, device="cuda")

Collected 24376 samples of dimension 512


In [16]:
analysis = analyze_subspace_dimensions(sub_model, device="cuda")


In [18]:
d = train_dataset[0][0].shape[0]

# ---- optimizer + scheduler ----
def make_optimizer_and_scheduler(model, train_loader, epochs=20, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler

def gap_to_float(x):
    if isinstance(x, dict):
        if "text_vision" in x:
            return float(x["text_vision"])
        return float(next(iter(x.values())))
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)

In [19]:
# =========================
# TRAIN ONLY on cuda:3
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = 1000
gaps = ["RMG", "L2M", "L2I", "cosineTP"]  # add cosineTP to the list of gaps to compute


# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0         # improvement if val_loss < best - min_delta_loss
min_delta_acc  = 0.0         # improvement if val_acc  > best + min_delta_acc
epochs_no_improve = 0

# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped


# -------------------------
# Helpers
# -------------------------
def gap_to_float(x):
    # supports dict + scalar tensors
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)


Using device: cuda:3 - NVIDIA RTX A6000


In [20]:
def run_epoch(dataloader, important_dims_joint, top_k, phase: str):
    """
    phase: "train" or "val"
    Returns: mean_loss, mean_acc, mean_lr (train only else None), mean_gaps dict
    """
    is_train = (phase == "train")
    if is_train:
        model_wrapper.train()
    else:
        model_wrapper.eval()

    losses, accs, lrs = [], [], []
    gaps_accum = {g: [] for g in gaps}

    for batch in tqdm(dataloader, desc=f"{phase.upper()}"):
        text_emb, vision_emb, labels = batch

        text_emb = text_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        vision_emb = vision_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        
        # Subspace Alignment
        with torch.no_grad():
            X, _, Y_al = apply_subspace_alignment(text_emb, vision_emb, sub_model, device=device, renorm=True)
            # Select only the important dimensions for both text and vision
            X = X[:, important_dims_joint]
            Y_al = Y_al[:, important_dims_joint]
            X = X.to(device)
            Y_al = Y_al.to(device)
            
        labels = labels.to(device=device, dtype=torch.long, non_blocking=True)
        labels = remap_labels(labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model_wrapper(X, Y_al)

        # Safety: label range
        num_classes = logits.size(1)
        y_min = int(labels.min().item())
        y_max = int(labels.max().item())
        if y_min < 0 or y_max >= num_classes:
            raise ValueError(
                f"[{phase.upper()}] Labels out of range after remap: "
                f"min={y_min}, max={y_max}, num_classes={num_classes}"
            )

        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            clip_grad_norm_(model_wrapper.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        acc = (logits.argmax(dim=1) == labels).float().mean()

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        if is_train:
            lrs.append(float(optimizer.param_groups[0]["lr"]))

        # ---- GAPs on GPU USING ALIGNED REPRESENTATIONS ----
        text_det = X.detach()
        vis_det  = Y_al.detach()
        for g in gaps:
            gaps_accum[g].append(gap_to_float(compute_gap(g, text_det, vis_det, iterations=None)))

    mean_loss = float(np.mean(losses)) if losses else float("nan")
    mean_acc  = float(np.mean(accs)) if accs else float("nan")
    mean_lr   = float(np.mean(lrs)) if (is_train and lrs) else None
    mean_gaps = {g: float(np.mean(gaps_accum[g])) for g in gaps}

    return mean_loss, mean_acc, mean_lr, mean_gaps


In [ ]:
# -------------------------
# Tracking
# -------------------------
epochs_results = {
    64: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
         "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    128: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    256: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    384: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
}

for selected_top_k in [64, 128, 256, 384]:

    print(f"\n\n=== Training with top-{selected_top_k} dimensions ===\n\n")

    important_dims_subspace = analysis[2][:selected_top_k].tolist()

    config = {
        "classifier": {
            "d": selected_top_k,
            "num_classes": len_classes,
        },
        "epochs": 1000,
        "batch_size": bs,
        "wandb": {
            "enabled": True,
            "project": "flickr30k_clip_classifier",
            "name": f"scenario_3.1_k{selected_top_k}",
        },
    }

    epochs = config["epochs"]

    # -------------------------
    # Save paths
    # -------------------------
    save_dir = config["wandb"]["name"]
    os.makedirs(save_dir, exist_ok=True)

    best_acc_path = os.path.join(save_dir, "best_model_val_acc.pt")
    best_loss_path = os.path.join(save_dir, "best_model_val_loss.pt")

    # -------------------------
    # Model
    # -------------------------
    model_wrapper = LinearProbing(
        d=config["classifier"]["d"],
        num_classes=config["classifier"]["num_classes"],
    ).to(device)

    # -------------------------
    # W&B Setup
    # -------------------------
    use_wandb = bool(config.get("wandb", {}).get("enabled", False))

    if use_wandb:
        import wandb
        wandb.init(
            project=config["wandb"]["project"],
            name=config["wandb"]["name"],
            config=config,
        )

    # -------------------------
    # Optimizer + Scheduler
    # -------------------------
    optimizer, scheduler = make_optimizer_and_scheduler(
        model_wrapper,
        train_dataloader,
        epochs=epochs,
        max_lr=3e-4,
        weight_decay=1e-4,
    )

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_loss_model_state_dict = None
    best_acc_model_state_dict = None

    epochs_no_improve = 0

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(epochs):

        train_loss, train_acc, train_lr, train_gaps = run_epoch(
            train_dataloader,
            important_dims_subspace,
            selected_top_k,
            phase="train",
        )

        val_loss, val_acc, _, val_gaps = run_epoch(
            test_dataloader,
            important_dims_subspace,
            selected_top_k,
            phase="val",
        )

        # -------------------------
        # Early stopping logic
        # -------------------------
        improved = False

        if val_loss < (best_val_loss - min_delta_loss):
            best_val_loss = val_loss
            best_loss_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_loss",
                    "metric_value": best_val_loss,
                    "model_state_dict": best_loss_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_loss_path,
            )

        if val_acc > (best_val_acc + min_delta_acc):
            best_val_acc = val_acc
            best_acc_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_accuracy",
                    "metric_value": best_val_acc,
                    "model_state_dict": best_acc_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_acc_path,
            )

        if improved:
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # -------------------------
        # W&B Logging
        # -------------------------
        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/acc": train_acc,
                "train/lr": train_lr,
                "train/RMG": train_gaps["RMG"],
                "train/L2M": train_gaps["L2M"],
                "train/L2I": train_gaps["L2I"],
                "train/cosineTP": train_gaps["cosineTP"],
                "val/loss": val_loss,
                "val/acc": val_acc,
                "val/RMG": val_gaps["RMG"],
                "val/L2M": val_gaps["L2M"],
                "val/L2I": val_gaps["L2I"],
                "val/cosineTP": val_gaps["cosineTP"],
                "early_stop/epochs_no_improve": epochs_no_improve,
                "best/val_loss": best_val_loss,
                "best/val_acc": best_val_acc,
            })

        if epochs_no_improve >= patience:
            print(f"\n[EARLY STOP] Stopped at epoch {epoch+1}")
            break

    # -------------------------
    # Finish W&B run properly
    # -------------------------
    if use_wandb:
        wandb.finish()

    print(f"\nTraining finished for top-{selected_top_k}")
    print("Best val loss:", best_val_loss)
    print("Best val acc :", best_val_acc)

    # -------------------------
    # Load best model
    # -------------------------
    if best_acc_model_state_dict is not None:
        model_wrapper.load_state_dict(best_acc_model_state_dict)
    elif best_loss_model_state_dict is not None:
        model_wrapper.load_state_dict(best_loss_model_state_dict)

    final_best_path = os.path.join(save_dir, "best_model_loaded_final.pt")
    torch.save(
        {
            "model_state_dict": model_wrapper.state_dict(),
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "config": config,
        },
        final_best_path,
    )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/emanuele/.netrc.




=== Training with top-64 dimensions ===




wandb: Currently logged in as: rucci-emanuele (rucci-emanuele-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


VAL: 100%|██████████| 3/3 [00:00<00:00,  7.22it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 201


...


Training finished for top-256
Best val loss: 4.534521579742432
Best val acc : 0.2953234513600667


=== Training with top-384 dimensions ===




VAL: 100%|██████████| 3/3 [00:00<00:00,  3.94it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 334


best/val_acc,▁▂▂▄▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████
best/val_loss,██████▇▇▇▆▆▆▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
early_stop/epochs_no_improve,▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▁▂▂▁▁▁▂▁▁▂▄▅▆▂▄▁▁▃▅▅▆▆▂▃█
epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█
train/L2I,▅▆▂▅▇▂▂▆▂▂▁▁▁▂▁▁▂▁▂▁▂▇▁▁▆▇▁▁▃▇▁▆▃▃█▁▁▁▁▁
train/L2M,▆▅▂█▂▂▁▆▆▂▆▂▁▁▁▁█▁▁▁▁▁▁▁▂▁▁▃▄▆▁▁▁▁▁▁▁▃▁▁
train/RMG,█▃▂▅█▆▂▇▂▁▇▆▇▃▁█▅▂▃█▁▁▆▂▁▁▁▁▁▁▃▃▁▁▁▁▁▁▁▁
train/acc,▁▁▁▁▂▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇█████████
train/cosineTP,▃▇▁▆▇▂▂▂▃▁█▆▂█▄▂█▁██▆▁▁██████▁█████▇████
train/loss,███████▇▇▆▆▆▆▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▂▁
+7,...



Training finished for top-384
Best val loss: 3.3000710010528564
Best val acc : 0.41964537898699444


## Scenario 3.2
We fit the linear probing layer on the first k dims by single modality importance selected by considering the subspace dimension = 256.

text_importance = k1_dim <br>
vision importance = k2_dim <br>
linear_probing(cat(text[k1_dim], vision[k2_dim]))


In [9]:
from subspace_alignment.subspace_alignment import fit_subspace_alignment, fit_subspace_alignment, analyze_subspace_dimensions, apply_subspace_alignment
with torch.no_grad():
    sub_model = fit_subspace_alignment(train_dataloader, d_sub=256, device="cuda")

Collected 24376 samples of dimension 512


In [10]:
analysis = analyze_subspace_dimensions(sub_model, device="cuda")


In [12]:
d = train_dataset[0][0].shape[0]

# ---- optimizer + scheduler ----
def make_optimizer_and_scheduler(model, train_loader, epochs=20, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler

def gap_to_float(x):
    if isinstance(x, dict):
        if "text_vision" in x:
            return float(x["text_vision"])
        return float(next(iter(x.values())))
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)

In [13]:
# =========================
# TRAIN ONLY on cuda:3
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = 1000
gaps = ["RMG", "L2M", "L2I", "cosineTP"]  # add cosineTP to the list of gaps to compute


# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0         # improvement if val_loss < best - min_delta_loss
min_delta_acc  = 0.0         # improvement if val_acc  > best + min_delta_acc
epochs_no_improve = 0

# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped


# -------------------------
# Helpers
# -------------------------
def gap_to_float(x):
    # supports dict + scalar tensors
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)


Using device: cuda:3 - NVIDIA RTX A6000


In [14]:
def run_epoch(dataloader, important_dims_X, important_dims_Y, top_k, phase: str):
    """
    phase: "train" or "val"
    Returns: mean_loss, mean_acc, mean_lr (train only else None), mean_gaps dict
    """
    is_train = (phase == "train")
    if is_train:
        model_wrapper.train()
    else:
        model_wrapper.eval()

    losses, accs, lrs = [], [], []
    gaps_accum = {g: [] for g in gaps}

    for batch in tqdm(dataloader, desc=f"{phase.upper()}"):
        text_emb, vision_emb, labels = batch

        text_emb = text_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        vision_emb = vision_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        
        # Subspace Alignment
        with torch.no_grad():
            X, _, Y_al = apply_subspace_alignment(text_emb, vision_emb, sub_model, device=device, renorm=True)
            # Select only the important dimensions for both text and vision in the aligned space
            X = X[:, important_dims_X]
            Y_al = Y_al[:, important_dims_Y]
            X = X.to(device)
            Y_al = Y_al.to(device)
            
        labels = labels.to(device=device, dtype=torch.long, non_blocking=True)
        labels = remap_labels(labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model_wrapper(X, Y_al)

        # Safety: label range
        num_classes = logits.size(1)
        y_min = int(labels.min().item())
        y_max = int(labels.max().item())
        if y_min < 0 or y_max >= num_classes:
            raise ValueError(
                f"[{phase.upper()}] Labels out of range after remap: "
                f"min={y_min}, max={y_max}, num_classes={num_classes}"
            )

        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            clip_grad_norm_(model_wrapper.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        acc = (logits.argmax(dim=1) == labels).float().mean()

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        if is_train:
            lrs.append(float(optimizer.param_groups[0]["lr"]))

        # ---- GAPs on GPU USING ALIGNED REPRESENTATIONS ----
        text_det = X.detach()
        vis_det  = Y_al.detach()
        for g in gaps:
            gaps_accum[g].append(gap_to_float(compute_gap(g, text_det, vis_det, iterations=None)))

    mean_loss = float(np.mean(losses)) if losses else float("nan")
    mean_acc  = float(np.mean(accs)) if accs else float("nan")
    mean_lr   = float(np.mean(lrs)) if (is_train and lrs) else None
    mean_gaps = {g: float(np.mean(gaps_accum[g])) for g in gaps}

    return mean_loss, mean_acc, mean_lr, mean_gaps


In [15]:
# -------------------------
# Tracking
# -------------------------
epochs_results = {
    64: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
         "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    128: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    256: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    384: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
}

for selected_top_k in [ 128, 256, 384]:

    print(f"\n\n=== Training with top-{selected_top_k} dimensions ===\n\n")

    important_dims_subspace_X = analysis[0][:selected_top_k].tolist()
    important_dims_subspace_Y = analysis[1][:selected_top_k].tolist()

    config = {
        "classifier": {
            "d": selected_top_k,
            "num_classes": len_classes,
        },
        "epochs": 1000,
        "batch_size": bs,
        "wandb": {
            "enabled": True,
            "project": "flickr30k_clip_classifier",
            "name": f"scenario_3.2_k{selected_top_k}",
        },
    }

    epochs = config["epochs"]

    # -------------------------
    # Save paths
    # -------------------------
    save_dir = config["wandb"]["name"]
    os.makedirs(save_dir, exist_ok=True)

    best_acc_path = os.path.join(save_dir, "best_model_val_acc.pt")
    best_loss_path = os.path.join(save_dir, "best_model_val_loss.pt")

    # -------------------------
    # Model
    # -------------------------
    model_wrapper = LinearProbing(
        d=config["classifier"]["d"],
        num_classes=config["classifier"]["num_classes"],
    ).to(device)

    # -------------------------
    # W&B Setup
    # -------------------------
    use_wandb = bool(config.get("wandb", {}).get("enabled", False))

    if use_wandb:
        import wandb
        wandb.init(
            project=config["wandb"]["project"],
            name=config["wandb"]["name"],
            config=config,
        )

    # -------------------------
    # Optimizer + Scheduler
    # -------------------------
    optimizer, scheduler = make_optimizer_and_scheduler(
        model_wrapper,
        train_dataloader,
        epochs=epochs,
        max_lr=3e-4,
        weight_decay=1e-4,
    )

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_loss_model_state_dict = None
    best_acc_model_state_dict = None

    epochs_no_improve = 0

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(epochs):

        train_loss, train_acc, train_lr, train_gaps = run_epoch(
            train_dataloader,
            important_dims_subspace_X,
            important_dims_subspace_Y,
            selected_top_k,
            phase="train",
        )

        val_loss, val_acc, _, val_gaps = run_epoch(
            test_dataloader,
            important_dims_subspace_X,
            important_dims_subspace_Y,
            selected_top_k,
            phase="val",
        )

        # -------------------------
        # Early stopping logic
        # -------------------------
        improved = False

        if val_loss < (best_val_loss - min_delta_loss):
            best_val_loss = val_loss
            best_loss_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_loss",
                    "metric_value": best_val_loss,
                    "model_state_dict": best_loss_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_loss_path,
            )

        if val_acc > (best_val_acc + min_delta_acc):
            best_val_acc = val_acc
            best_acc_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_accuracy",
                    "metric_value": best_val_acc,
                    "model_state_dict": best_acc_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_acc_path,
            )

        if improved:
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # -------------------------
        # W&B Logging
        # -------------------------
        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/acc": train_acc,
                "train/lr": train_lr,
                "train/RMG": train_gaps["RMG"],
                "train/L2M": train_gaps["L2M"],
                "train/L2I": train_gaps["L2I"],
                "train/cosineTP": train_gaps["cosineTP"],
                "val/loss": val_loss,
                "val/acc": val_acc,
                "val/RMG": val_gaps["RMG"],
                "val/L2M": val_gaps["L2M"],
                "val/L2I": val_gaps["L2I"],
                "val/cosineTP": val_gaps["cosineTP"],
                "early_stop/epochs_no_improve": epochs_no_improve,
                "best/val_loss": best_val_loss,
                "best/val_acc": best_val_acc,
            })

        if epochs_no_improve >= patience:
            print(f"\n[EARLY STOP] Stopped at epoch {epoch+1}")
            break

    # -------------------------
    # Finish W&B run properly
    # -------------------------
    if use_wandb:
        wandb.finish()

    print(f"\nTraining finished for top-{selected_top_k}")
    print("Best val loss:", best_val_loss)
    print("Best val acc :", best_val_acc)

    # -------------------------
    # Load best model
    # -------------------------
    if best_acc_model_state_dict is not None:
        model_wrapper.load_state_dict(best_acc_model_state_dict)
    elif best_loss_model_state_dict is not None:
        model_wrapper.load_state_dict(best_loss_model_state_dict)

    final_best_path = os.path.join(save_dir, "best_model_loaded_final.pt")
    torch.save(
        {
            "model_state_dict": model_wrapper.state_dict(),
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "config": config,
        },
        final_best_path,
    )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/emanuele/.netrc.




=== Training with top-128 dimensions ===




wandb: Currently logged in as: rucci-emanuele (rucci-emanuele-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


VAL: 100%|██████████| 3/3 [00:00<00:00,  6.29it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 197


...


Training finished for top-256
Best val loss: 4.281664530436198
Best val acc : 0.3116154472033183


=== Training with top-384 dimensions ===




VAL: 100%|██████████| 3/3 [00:00<00:00,  3.98it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 180


best/val_acc,▁▁▁▁▂▂▃▄▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████
best/val_loss,███████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁
early_stop/epochs_no_improve,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▅█
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█
train/L2I,▅▃▂▄▂█▂█▂▂▂▃▁▂▂▂▂▂▁▂▂▁▂▂▂▂▁▁▂▂▁▂▂▂▂▂▂▁▂▁
train/L2M,▁▄█▂▂▁▄▁▁▁▁▂▂▂▁▁▂▂▂▂▁▂▂▂▂▃▂▂▂▂▂▂▃▂▁▁▁▁▁▁
train/RMG,▁▇▅▄█▄▄▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆
train/acc,▁▁▁▂▂▃▄▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇▇█████
train/cosineTP,▅▄▆█▄▆▂▂▆▃▃▃▁▂▂▂▁▂▃▂▂▃▂▂▃▁▂▂▂▁▂▂▂▂▃▃▃▃▂▃
train/loss,███████▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁
+7,...



Training finished for top-384
Best val loss: 4.0652337074279785
Best val acc : 0.33917463819185895


## Scenario 4.1
We fit the linear probing layer on the first k-dims dim by joint importance selecting a subspace = 256

Here we use the values of the dimension taking them from the original space.

In [9]:
from subspace_alignment.subspace_alignment import fit_subspace_alignment, fit_subspace_alignment, analyze_subspace_dimensions, apply_subspace_alignment
with torch.no_grad():
    sub_model = fit_subspace_alignment(train_dataloader, d_sub=256, device="cuda")

Collected 24376 samples of dimension 512


In [10]:
analysis = analyze_subspace_dimensions(sub_model, device="cuda")

In [11]:
d = train_dataset[0][0].shape[0]

# ---- optimizer + scheduler ----
def make_optimizer_and_scheduler(model, train_loader, epochs=20, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler

def gap_to_float(x):
    if isinstance(x, dict):
        if "text_vision" in x:
            return float(x["text_vision"])
        return float(next(iter(x.values())))
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)

In [12]:
# =========================
# TRAIN ONLY on cuda:3
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = 1000
gaps = ["RMG", "L2M", "L2I", "cosineTP"]  # add cosineTP to the list of gaps to compute


# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0         # improvement if val_loss < best - min_delta_loss
min_delta_acc  = 0.0         # improvement if val_acc  > best + min_delta_acc
epochs_no_improve = 0

# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped


# -------------------------
# Helpers
# -------------------------
def gap_to_float(x):
    # supports dict + scalar tensors
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)


Using device: cuda:3 - NVIDIA RTX A6000


In [13]:
def run_epoch(dataloader, important_dims_joint, top_k, phase: str):
    """
    phase: "train" or "val"
    Returns: mean_loss, mean_acc, mean_lr (train only else None), mean_gaps dict
    """
    is_train = (phase == "train")
    if is_train:
        model_wrapper.train()
    else:
        model_wrapper.eval()

    losses, accs, lrs = [], [], []
    gaps_accum = {g: [] for g in gaps}

    for batch in tqdm(dataloader, desc=f"{phase.upper()}"):
        text_emb, vision_emb, labels = batch

        text_emb = text_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        vision_emb = vision_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        
        # Select duns
        text_emb = text_emb[:, important_dims_joint]
        vision_emb = vision_emb[:, important_dims_joint]
            
        labels = labels.to(device=device, dtype=torch.long, non_blocking=True)
        labels = remap_labels(labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model_wrapper(text_emb, vision_emb)

        # Safety: label range
        num_classes = logits.size(1)
        y_min = int(labels.min().item())
        y_max = int(labels.max().item())
        if y_min < 0 or y_max >= num_classes:
            raise ValueError(
                f"[{phase.upper()}] Labels out of range after remap: "
                f"min={y_min}, max={y_max}, num_classes={num_classes}"
            )

        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            clip_grad_norm_(model_wrapper.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        acc = (logits.argmax(dim=1) == labels).float().mean()

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        if is_train:
            lrs.append(float(optimizer.param_groups[0]["lr"]))

        # ---- GAPs on GPU USING ALIGNED REPRESENTATIONS ----
        text_det = text_emb.detach()
        vis_det  = vision_emb.detach()
        for g in gaps:
            gaps_accum[g].append(gap_to_float(compute_gap(g, text_det, vis_det, iterations=None)))

    mean_loss = float(np.mean(losses)) if losses else float("nan")
    mean_acc  = float(np.mean(accs)) if accs else float("nan")
    mean_lr   = float(np.mean(lrs)) if (is_train and lrs) else None
    mean_gaps = {g: float(np.mean(gaps_accum[g])) for g in gaps}

    return mean_loss, mean_acc, mean_lr, mean_gaps


In [14]:
# -------------------------
# Tracking
# -------------------------
epochs_results = {
    64: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
         "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    128: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    256: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    384: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
}

for selected_top_k in [64, 128, 256, 384]:

    print(f"\n\n=== Training with top-{selected_top_k} dimensions ===\n\n")

    important_dims_subspace = analysis[2][:selected_top_k].tolist()

    config = {
        "classifier": {
            "d": selected_top_k,
            "num_classes": len_classes,
        },
        "epochs": 1000,
        "batch_size": bs,
        "wandb": {
            "enabled": True,
            "project": "flickr30k_clip_classifier",
            "name": f"scenario_4.1_k{selected_top_k}",
        },
    }

    epochs = config["epochs"]

    # -------------------------
    # Save paths
    # -------------------------
    save_dir = config["wandb"]["name"]
    os.makedirs(save_dir, exist_ok=True)

    best_acc_path = os.path.join(save_dir, "best_model_val_acc.pt")
    best_loss_path = os.path.join(save_dir, "best_model_val_loss.pt")

    # -------------------------
    # Model
    # -------------------------
    model_wrapper = LinearProbing(
        d=config["classifier"]["d"],
        num_classes=config["classifier"]["num_classes"],
    ).to(device)

    # -------------------------
    # W&B Setup
    # -------------------------
    use_wandb = bool(config.get("wandb", {}).get("enabled", False))

    if use_wandb:
        import wandb
        wandb.init(
            project=config["wandb"]["project"],
            name=config["wandb"]["name"],
            config=config,
        )

    # -------------------------
    # Optimizer + Scheduler
    # -------------------------
    optimizer, scheduler = make_optimizer_and_scheduler(
        model_wrapper,
        train_dataloader,
        epochs=epochs,
        max_lr=3e-4,
        weight_decay=1e-4,
    )

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_loss_model_state_dict = None
    best_acc_model_state_dict = None

    epochs_no_improve = 0

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(epochs):

        train_loss, train_acc, train_lr, train_gaps = run_epoch(
            train_dataloader,
            important_dims_subspace,
            selected_top_k,
            phase="train",
        )

        val_loss, val_acc, _, val_gaps = run_epoch(
            test_dataloader,
            important_dims_subspace,
            selected_top_k,
            phase="val",
        )

        # -------------------------
        # Early stopping logic
        # -------------------------
        improved = False

        if val_loss < (best_val_loss - min_delta_loss):
            best_val_loss = val_loss
            best_loss_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_loss",
                    "metric_value": best_val_loss,
                    "model_state_dict": best_loss_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_loss_path,
            )

        if val_acc > (best_val_acc + min_delta_acc):
            best_val_acc = val_acc
            best_acc_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_accuracy",
                    "metric_value": best_val_acc,
                    "model_state_dict": best_acc_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_acc_path,
            )

        if improved:
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # -------------------------
        # W&B Logging
        # -------------------------
        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/acc": train_acc,
                "train/lr": train_lr,
                "train/RMG": train_gaps["RMG"],
                "train/L2M": train_gaps["L2M"],
                "train/L2I": train_gaps["L2I"],
                "train/cosineTP": train_gaps["cosineTP"],
                "val/loss": val_loss,
                "val/acc": val_acc,
                "val/RMG": val_gaps["RMG"],
                "val/L2M": val_gaps["L2M"],
                "val/L2I": val_gaps["L2I"],
                "val/cosineTP": val_gaps["cosineTP"],
                "early_stop/epochs_no_improve": epochs_no_improve,
                "best/val_loss": best_val_loss,
                "best/val_acc": best_val_acc,
            })

        if epochs_no_improve >= patience:
            print(f"\n[EARLY STOP] Stopped at epoch {epoch+1}")
            break

    # -------------------------
    # Finish W&B run properly
    # -------------------------
    if use_wandb:
        wandb.finish()

    print(f"\nTraining finished for top-{selected_top_k}")
    print("Best val loss:", best_val_loss)
    print("Best val acc :", best_val_acc)

    # -------------------------
    # Load best model
    # -------------------------
    if best_acc_model_state_dict is not None:
        model_wrapper.load_state_dict(best_acc_model_state_dict)
    elif best_loss_model_state_dict is not None:
        model_wrapper.load_state_dict(best_loss_model_state_dict)

    final_best_path = os.path.join(save_dir, "best_model_loaded_final.pt")
    torch.save(
        {
            "model_state_dict": model_wrapper.state_dict(),
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "config": config,
        },
        final_best_path,
    )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/emanuele/.netrc.




=== Training with top-64 dimensions ===




wandb: Currently logged in as: rucci-emanuele (rucci-emanuele-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


VAL: 100%|██████████| 3/3 [00:00<00:00, 25.78it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 567


...


Training finished for top-256
Best val loss: 2.8717169761657715
Best val acc : 0.481083482503891


=== Training with top-384 dimensions ===




VAL: 100%|██████████| 3/3 [00:00<00:00,  6.46it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 201


best/val_acc,▁▁▁▂▂▃▄▅▅▅▆▆▆▇▇▇▇▇██████████████████████
best/val_loss,███▇▇▇▆▆▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
early_stop/epochs_no_improve,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▄▄▅▅█
epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/L2I,▄▂▅▆▆▃▅▅▄▇▃▄▁▃▆▃▃▃█▄▅▂▃▃▄▁▄▂▃▆▅▃▂▃▅▆▆▃▂▅
train/L2M,▄▆▄▄▄▅▆▂▁▃▆▁▆▄▄▄▅▆█▃▄▄▄▃▆▁▅▃▅▂▅▄▄▂▆▇▅▄▇▄
train/RMG,▅▃▆▃▄▅▃▅▂▄▃▄▁▃▅▁▂▄▃▃▆▄▃▃▄█▄▄▄▅█▂▆▇▇▄▄▆▃▄
train/acc,▁▁▁▂▂▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████
train/cosineTP,▃▄▆▅▄▃▅▃▅▂▄▄▄▅▃▆▆▇▅▇▄▄▅▂▄█▅▅▃▁▆▆▇▆▅▅▁▇█▅
train/loss,████▇▇▇▇▆▆▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+7,...



Training finished for top-384
Best val loss: 2.830925782521566
Best val acc : 0.4857046107451121


## Scenario 4.2
We fit the linear probing layer on the first k-dims dim by importance for each modality selecting a subspace = 256

Here we use the values of the dimension taking them from the original space.

In [15]:
from subspace_alignment.subspace_alignment import fit_subspace_alignment, fit_subspace_alignment, analyze_subspace_dimensions, apply_subspace_alignment
with torch.no_grad():
    sub_model = fit_subspace_alignment(train_dataloader, d_sub=256, device="cuda")

Collected 24376 samples of dimension 512


In [16]:
analysis = analyze_subspace_dimensions(sub_model, device="cuda")

In [17]:
d = train_dataset[0][0].shape[0]

# ---- optimizer + scheduler ----
def make_optimizer_and_scheduler(model, train_loader, epochs=20, max_lr=3e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=25.0,
        final_div_factor=1e4,
        anneal_strategy="cos",
    )
    return optimizer, scheduler

def gap_to_float(x):
    if isinstance(x, dict):
        if "text_vision" in x:
            return float(x["text_vision"])
        return float(next(iter(x.values())))
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)

In [18]:
# =========================
# TRAIN ONLY on cuda:3
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = 1000
gaps = ["RMG", "L2M", "L2I", "cosineTP"]  # add cosineTP to the list of gaps to compute


# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0         # improvement if val_loss < best - min_delta_loss
min_delta_acc  = 0.0         # improvement if val_acc  > best + min_delta_acc
epochs_no_improve = 0

# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped


# -------------------------
# Helpers
# -------------------------
def gap_to_float(x):
    # supports dict + scalar tensors
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)


Using device: cuda:3 - NVIDIA RTX A6000


In [19]:
def run_epoch(dataloader, important_dims_X, important_dims_Y, top_k, phase: str):
    """
    phase: "train" or "val"
    Returns: mean_loss, mean_acc, mean_lr (train only else None), mean_gaps dict
    """
    is_train = (phase == "train")
    if is_train:
        model_wrapper.train()
    else:
        model_wrapper.eval()

    losses, accs, lrs = [], [], []
    gaps_accum = {g: [] for g in gaps}

    for batch in tqdm(dataloader, desc=f"{phase.upper()}"):
        text_emb, vision_emb, labels = batch

        text_emb = text_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        vision_emb = vision_emb.to(device=device, dtype=torch.float32, non_blocking=True)
        
        # Select important dims per modality in the original space
        text_emb = text_emb[:, important_dims_X]
        vision_emb = vision_emb[:, important_dims_Y]
        
            
        labels = labels.to(device=device, dtype=torch.long, non_blocking=True)
        labels = remap_labels(labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model_wrapper(text_emb, vision_emb)

        # Safety: label range
        num_classes = logits.size(1)
        y_min = int(labels.min().item())
        y_max = int(labels.max().item())
        if y_min < 0 or y_max >= num_classes:
            raise ValueError(
                f"[{phase.upper()}] Labels out of range after remap: "
                f"min={y_min}, max={y_max}, num_classes={num_classes}"
            )

        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            clip_grad_norm_(model_wrapper.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        acc = (logits.argmax(dim=1) == labels).float().mean()

        losses.append(float(loss.item()))
        accs.append(float(acc.item()))
        if is_train:
            lrs.append(float(optimizer.param_groups[0]["lr"]))

        # ---- GAPs on GPU USING ALIGNED REPRESENTATIONS ----
        text_det = text_emb.detach()
        vis_det  = vision_emb.detach()
        for g in gaps:
            gaps_accum[g].append(gap_to_float(compute_gap(g, text_det, vis_det, iterations=None)))

    mean_loss = float(np.mean(losses)) if losses else float("nan")
    mean_acc  = float(np.mean(accs)) if accs else float("nan")
    mean_lr   = float(np.mean(lrs)) if (is_train and lrs) else None
    mean_gaps = {g: float(np.mean(gaps_accum[g])) for g in gaps}

    return mean_loss, mean_acc, mean_lr, mean_gaps


In [20]:
# -------------------------
# Tracking
# -------------------------
epochs_results = {
    64: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
         "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    128: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    256: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
    384: {"train": {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": [], "lr": []},
          "val":   {"loss": [], "accuracy": [], "RMG": [], "L2M": [], "L2I": [], "cosineTP": []}},
}

for selected_top_k in [64, 128, 256, 384]:

    print(f"\n\n=== Training with top-{selected_top_k} dimensions ===\n\n")

    important_dims_subspace_X = analysis[0][:selected_top_k].tolist()
    important_dims_subspace_Y = analysis[1][:selected_top_k].tolist()

    config = {
        "classifier": {
            "d": selected_top_k,
            "num_classes": len_classes,
        },
        "epochs": 1000,
        "batch_size": bs,
        "wandb": {
            "enabled": True,
            "project": "flickr30k_clip_classifier",
            "name": f"scenario_4.2_k{selected_top_k}",
        },
    }

    epochs = config["epochs"]

    # -------------------------
    # Save paths
    # -------------------------
    save_dir = config["wandb"]["name"]
    os.makedirs(save_dir, exist_ok=True)

    best_acc_path = os.path.join(save_dir, "best_model_val_acc.pt")
    best_loss_path = os.path.join(save_dir, "best_model_val_loss.pt")

    # -------------------------
    # Model
    # -------------------------
    model_wrapper = LinearProbing(
        d=config["classifier"]["d"],
        num_classes=config["classifier"]["num_classes"],
    ).to(device)

    # -------------------------
    # W&B Setup
    # -------------------------
    use_wandb = bool(config.get("wandb", {}).get("enabled", False))

    if use_wandb:
        import wandb
        wandb.init(
            project=config["wandb"]["project"],
            name=config["wandb"]["name"],
            config=config,
        )

    # -------------------------
    # Optimizer + Scheduler
    # -------------------------
    optimizer, scheduler = make_optimizer_and_scheduler(
        model_wrapper,
        train_dataloader,
        epochs=epochs,
        max_lr=3e-4,
        weight_decay=1e-4,
    )

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_loss_model_state_dict = None
    best_acc_model_state_dict = None

    epochs_no_improve = 0

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(epochs):

        train_loss, train_acc, train_lr, train_gaps = run_epoch(
            train_dataloader,
            important_dims_subspace_X,
            important_dims_subspace_Y,
            selected_top_k,
            phase="train",
        )

        val_loss, val_acc, _, val_gaps = run_epoch(
            test_dataloader,
            important_dims_subspace_X,
            important_dims_subspace_Y,
            selected_top_k,
            phase="val",
        )

        # -------------------------
        # Early stopping logic
        # -------------------------
        improved = False

        if val_loss < (best_val_loss - min_delta_loss):
            best_val_loss = val_loss
            best_loss_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_loss",
                    "metric_value": best_val_loss,
                    "model_state_dict": best_loss_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_loss_path,
            )

        if val_acc > (best_val_acc + min_delta_acc):
            best_val_acc = val_acc
            best_acc_model_state_dict = copy.deepcopy(model_wrapper.state_dict())
            improved = True

            torch.save(
                {
                    "epoch": epoch + 1,
                    "metric": "val_accuracy",
                    "metric_value": best_val_acc,
                    "model_state_dict": best_acc_model_state_dict,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                },
                best_acc_path,
            )

        if improved:
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # -------------------------
        # W&B Logging
        # -------------------------
        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/acc": train_acc,
                "train/lr": train_lr,
                "train/RMG": train_gaps["RMG"],
                "train/L2M": train_gaps["L2M"],
                "train/L2I": train_gaps["L2I"],
                "train/cosineTP": train_gaps["cosineTP"],
                "val/loss": val_loss,
                "val/acc": val_acc,
                "val/RMG": val_gaps["RMG"],
                "val/L2M": val_gaps["L2M"],
                "val/L2I": val_gaps["L2I"],
                "val/cosineTP": val_gaps["cosineTP"],
                "early_stop/epochs_no_improve": epochs_no_improve,
                "best/val_loss": best_val_loss,
                "best/val_acc": best_val_acc,
            })

        if epochs_no_improve >= patience:
            print(f"\n[EARLY STOP] Stopped at epoch {epoch+1}")
            break

    # -------------------------
    # Finish W&B run properly
    # -------------------------
    if use_wandb:
        wandb.finish()

    print(f"\nTraining finished for top-{selected_top_k}")
    print("Best val loss:", best_val_loss)
    print("Best val acc :", best_val_acc)

    # -------------------------
    # Load best model
    # -------------------------
    if best_acc_model_state_dict is not None:
        model_wrapper.load_state_dict(best_acc_model_state_dict)
    elif best_loss_model_state_dict is not None:
        model_wrapper.load_state_dict(best_loss_model_state_dict)

    final_best_path = os.path.join(save_dir, "best_model_loaded_final.pt")
    torch.save(
        {
            "model_state_dict": model_wrapper.state_dict(),
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "config": config,
        },
        final_best_path,
    )



=== Training with top-64 dimensions ===




VAL: 100%|██████████| 3/3 [00:00<00:00, 24.11it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 587


best/val_acc,▁▁▂▂▃▄▅▅▅▆▇▇▇▇▇▇▇▇▇▇████████████████████
best/val_loss,█▇▇▅▅▄▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
early_stop/epochs_no_improve,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▃█
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇██
train/L2I,█▆▃▂▄▇▄▃▇▅▅▂▅▅▂▇▅▄▁▄▁▃▃▅▁▃▄▁▃▅▄▅█▁▄▃▄▆▄▄
train/L2M,▄▆▅▇▃▄▅▇▃▁▅▅▂▅▄▆▄▅▄▃▁▇█▆▃▄▇▃█▃█▅▆▄▅▅▄▆▇▆
train/RMG,▇▃▆█▄▂▃▅▄▆▅▆▂▆▂█▁▆▆▆▅▃▅▃▆▆█▄▇▃▄▄▃▆▃█▆▄▃▇
train/acc,▁▁▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████
train/cosineTP,▅▃▃▄▃▆▂▄▄▅▂▂▃▁▅▄▄▅▄▃▃▆▄▄▃▅▅▁▄▇▃▅▄▄▃▅▆█▅▅
train/loss,██▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+7,...



Training finished for top-64
Best val loss: 3.244706948598226
Best val acc : 0.40904518961906433


=== Training with top-128 dimensions ===




...


Training finished for top-256
Best val loss: 2.8703420162200928
Best val acc : 0.47516825795173645


=== Training with top-384 dimensions ===




VAL: 100%|██████████| 3/3 [00:00<00:00,  6.41it/s]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



[EARLY STOP] Stopped at epoch 211


best/val_acc,▁▁▁▁▂▂▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇████████████████
best/val_loss,██▇▇▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
early_stop/epochs_no_improve,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▄▅█
epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/L2I,▅▄▆▁▂▆▁▄▅▇▄▂▅▄▆▃▅▇▃▆▆▅▄▄▆▆▄▅▃▇▄▇▇▃█▇▇▇▅▂
train/L2M,▅▅▄▃▄▆▃▄▆▇▄▁▄▇█▃▅▅▆▄▄▄▃▅▇▃▄▇▂▅▁█▂▅▄▃▆▅▄▇
train/RMG,▄▂▄▃▆▅▂▂▄▃▄▅▃▄▁▇▁▃▄▄▃▃▄▅▃▅█▄▅▃▆▂▃▄▃▆▃▂▆▆
train/acc,▁▁▁▂▂▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████████
train/cosineTP,▅▃▄▄▆▆▄▄▅▅▆▅▄▄▂▇▃▄▅▃▄▅▆▂▁▄▅▂▅▆▄▄▅▄▆▄█▂▅▃
train/loss,██▇▇▇▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
+7,...



Training finished for top-384
Best val loss: 2.832225720087687
Best val acc : 0.483402023712794


## Scenario 5.1: First k dims (Original space - no alignment)
We fit the linear probing layer on the **first k dimensions** (0, 1, 2, ..., k-1) from the original embedding space without any subspace alignment.

# Test
Now that we have the trained models and we also know from the training set which are the best semantic dimensions, we can test on the test set the classification results.

We also perform a test in which we give to the classifiers k random dims or first k dim in the original space.

In [ ]:
# =========================
# Scenario 5.1: First k dims (Original space - no alignment)
# =========================
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F

# -------------------------
# Device: cuda:3
# -------------------------
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() > 3, f"cuda:3 not available. device_count={torch.cuda.device_count()}"

device = torch.device("cuda:3")
torch.cuda.set_device(device)
print("Using device:", device, "-", torch.cuda.get_device_name(3))

# -------------------------
# Loss
# -------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)

# -------------------------
# Config
# -------------------------
epochs = 1000
gaps = ["RMG", "L2M", "L2I", "cosineTP"]

# -------------------------
# Early stopping (ON VALIDATION)
# -------------------------
patience = 30
min_delta_loss = 0.0
min_delta_acc = 0.0

# -------------------------
# Label remap: vectorized (GPU)
# -------------------------
max_orig_label = int(max(filtered_class2idx.keys()))
lut = torch.full((max_orig_label + 1,), -1, dtype=torch.long)
for k, v in filtered_class2idx.items():
    lut[int(k)] = int(v)
lut = lut.to(device)

def remap_labels(labels_gpu: torch.Tensor) -> torch.Tensor:
    if labels_gpu.max().item() >= lut.numel() or labels_gpu.min().item() < 0:
        raise ValueError(
            f"Original labels out of LUT range. min={labels_gpu.min().item()} "
            f"max={labels_gpu.max().item()} lut_size={lut.numel()}"
        )
    mapped = lut[labels_gpu]
    if (mapped < 0).any():
        bad = labels_gpu[mapped < 0][:20].detach().cpu().numpy()
        raise ValueError(f"Found labels not in filtered_class2idx. Examples: {bad}")
    return mapped

def gap_to_float(x):
    if isinstance(x, dict):
        v = x.get("text_vision", next(iter(x.values())))
        if isinstance(v, torch.Tensor):
            return float(v.item())
        return float(v)
    if isinstance(x, torch.Tensor):
        return float(x.item())
    if hasattr(x, "item"):
        return float(x.item())
    return float(x)